# 👁️ Transformers Visually: RNN vs Attention, Real Heatmaps, Training vs Inference

A **concepts-first, watch-it-happen** companion to `11_Transformers_Attention_Zero_to_Hero`.
That lab built the attention *formula* from scratch. This one answers the questions that
formula alone doesn't: **why** Transformers replaced RNNs, what attention actually *looks
like* on a real sentence, and how the *same* architecture behaves differently during
training vs. inference.

Same philosophy as the micrograd lab: build small honest pieces yourself, **render what's
actually happening** (attention weights as real heatmaps, not just numbers), and verify your
from-scratch code against a real framework at the end.

**Prerequisite:** `11_Transformers_Attention_Zero_to_Hero` (Q/K/V, the attention formula) and
ideally `21_Micrograd_Visual_Backprop` (forward/backward pass fundamentals).

**How this lab works** — 📖 Theory → 🧠 Mental model → 🖼️ Rendered visual (real heatmaps) →
🔬 Worked example → ⚡ Pro tips → ⚠️ Traps → ✏️ Your Turn → ✅ Solution.

**Roadmap**
1. Why Transformers replaced RNNs — build both, watch the difference happen
2. How it works end-to-end — real matrix shapes, visualized at every step
3. Multi-head attention, deep dive — render each head's pattern separately
4. Why attention works — a real coreference example, visualized
5. Training-time behavior — teacher forcing, one parallel pass
6. Inference-time behavior — autoregressive generation, KV-caching
7. 🏆 Capstone: verify your attention matches PyTorch's real `nn.MultiheadAttention` exactly


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
print("Ready.")

---
## Chapter 1 — Why Transformers Replaced RNNs (Build Both, Watch the Difference)

📖 **Theory.** An RNN processes a sequence one step at a time, updating a single **hidden
state** that must summarize *everything seen so far* in a fixed-size vector. A Transformer's
attention lets every position look directly at every other position, with no bottleneck.

This isn't a claim to take on faith — let's build the smallest possible RNN and *measure*
what happens to information over a sequence, honestly.

🧠 **Mental model.** An RNN's hidden state is like relaying a message person-to-person down a
long line — by the end, earlier details are faded or gone. Attention is everyone in the room
being able to turn and ask the original speaker directly, at any time.


In [ ]:
np.random.seed(0)
hidden_size = 8

def rnn_step(h, x, Wh, Wx):
    return np.tanh(Wh @ h + Wx @ x)

Wh = np.random.randn(hidden_size, hidden_size) * 0.3
Wx = np.random.randn(hidden_size, hidden_size) * 0.3

# Put a distinct "signal" in the very FIRST input, then feed 30 steps of unrelated
# "noise" inputs after it. Does the signal survive in the hidden state?
signal = np.random.randn(hidden_size)
h = np.zeros(hidden_size)
h = rnn_step(h, signal, Wh, Wx)
hidden_right_after_signal = h.copy()

seq_len = 30
history = [hidden_right_after_signal]
for t in range(1, seq_len):
    noise = np.random.randn(hidden_size) * 0.5
    h = rnn_step(h, noise, Wh, Wx)
    history.append(h.copy())

# cosine similarity between the hidden state right after the signal, and every later step
sims = [np.dot(hidden_right_after_signal, hh) /
        (np.linalg.norm(hidden_right_after_signal)*np.linalg.norm(hh) + 1e-9) for hh in history]

plt.figure(figsize=(7,3))
plt.plot(sims, marker='o', markersize=3)
plt.axhline(0, color='gray', linewidth=0.8)
plt.xlabel("steps since the signal"); plt.ylabel("similarity to original signal's hidden state")
plt.title("RNN hidden state: how much of the original signal survives?")
plt.show()
print(f"similarity after {seq_len} steps: {sims[-1]:.4f}  (1.0 = perfectly preserved, 0 = gone)")

⚠️ **This is a real, measured effect, not a metaphor.** By step 30, the hidden state's
similarity to its state right after the signal drops to near zero (often even slightly
negative) — the RNN's fixed-size hidden state genuinely cannot hold onto old information
indefinitely, no matter how important it was. This is the concrete mechanism behind
"vanishing gradients over time" and "RNNs struggle with long-range dependencies."

Now the contrast: attention doesn't route information *through* a bottleneck at all.


In [ ]:
def cosine_sim(a, b):
    return np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-9)

def attention_direct_access(query_position, keys, values):
    """With attention, ANY position can attend directly to ANY other position --
    there's no relay chain to travel down and no fixed-size bottleneck to squeeze through."""
    scores = keys @ query_position
    weights = np.exp(scores - scores.max())
    weights /= weights.sum()
    return weights @ values, weights

# Same setup: a "signal" at position 0, followed by 30 unrelated positions.
np.random.seed(0)
seq_len = 31
X = np.random.randn(seq_len, hidden_size) * 0.5
X[0] = signal   # same signal as before, at position 0

# position 30 (the LAST position) attending back across the whole sequence, including position 0
last_position_query = X[30]
result, attn_weights = attention_direct_access(last_position_query, X, X)

print(f"attention weight on position 0 (the original signal), from 30 steps away: {attn_weights[0]:.4f}")
print(f"(compare: an RNN's hidden state similarity to the same signal, same distance, was {sims[-1]:.4f})")
print("\nAttention doesn't need the signal to SURVIVE a relay chain -- it can look directly at position 0, always.")

⚡ **Pro tip.** This is also *why* Transformers train faster on modern hardware: an RNN's
steps are **sequential** (step 30 needs step 29's result first) — you cannot parallelize
across time. Attention computes every position's relationship to every other position as one
big matrix multiplication — **fully parallel**, which is exactly what GPUs are built for.

### ✏️ Your Turn 1.1
Increase `seq_len` in the RNN experiment to 60 (double it) and re-run. Does the final
similarity get better, worse, or about the same? What does that tell you about RNNs on
longer sequences?

In [ ]:
# re-run the RNN experiment with seq_len=60


✅ **Solution**
```python
# Similarity gets even closer to zero (or more negative) -- longer sequences make the
# problem WORSE, since there are more relay steps for the original signal to fade across.
# This is exactly why RNNs struggle disproportionately on long documents/conversations.
```

---
## Chapter 2 — How It Works End-to-End: Real Matrix Shapes, Visualized

📖 **Theory.** Let's trace one real (short) sentence through every stage of a Transformer
block, printing AND visualizing the actual matrix at each step — not just describing shapes
in words. This directly mirrors how the "Transformers Explained Visually" article walks
through internal operations end-to-end.

🖼️ We'll render each intermediate matrix as a heatmap so you can literally see values change
shape and content as data flows through the block.


In [ ]:
sentence = ["The", "cat", "sat", "on", "the", "mat"]
seq_len = len(sentence)
d_model = 8

# Stage 1: token embeddings (normally learned; random here, but real shape/role)
np.random.seed(5)
vocab = sorted(set(sentence))
embed_table = {w: np.random.randn(d_model)*0.3 for w in vocab}
X = np.array([embed_table[w] for w in sentence])   # (seq_len, d_model)
print("Stage 1 -- embeddings shape:", X.shape)

plt.figure(figsize=(6,3))
plt.imshow(X, cmap='RdBu', aspect='auto')
plt.yticks(range(seq_len), sentence)
plt.xlabel("embedding dimension"); plt.title("Stage 1: token embeddings")
plt.colorbar(label="value")
plt.show()

In [ ]:
# Stage 2: Q, K, V projections
Wq = np.random.randn(d_model, d_model)*0.3
Wk = np.random.randn(d_model, d_model)*0.3
Wv = np.random.randn(d_model, d_model)*0.3
Q, K_, V = X @ Wq, X @ Wk, X @ Wv
print("Stage 2 -- Q, K, V shapes:", Q.shape, K_.shape, V.shape, " (each still seq_len x d_model)")

# Stage 3: attention scores (seq_len x seq_len -- THIS is the key shape change)
scores = Q @ K_.T / np.sqrt(d_model)
weights = np.exp(scores - scores.max(axis=1, keepdims=True))
weights /= weights.sum(axis=1, keepdims=True)
print("Stage 3 -- attention weights shape:", weights.shape, " (seq_len x seq_len -- every word's relation to every other)")

plt.figure(figsize=(5,4))
plt.imshow(weights, cmap='viridis')
plt.xticks(range(seq_len), sentence, rotation=45); plt.yticks(range(seq_len), sentence)
plt.title("Stage 3: attention weights (row = query word, col = key word)")
plt.colorbar(label="attention weight")
plt.show()

In [ ]:
# Stage 4: weighted combination of Values -> output (BACK to seq_len x d_model)
output = weights @ V
print("Stage 4 -- output shape:", output.shape, " (same shape as input -- this is why blocks can STACK)")

plt.figure(figsize=(6,3))
plt.imshow(output, cmap='RdBu', aspect='auto')
plt.yticks(range(seq_len), sentence)
plt.xlabel("dimension"); plt.title("Stage 4: attention output (ready for the next layer, or to stack another block)")
plt.colorbar(label="value")
plt.show()

⚠️ **The one shape to remember.** Embeddings go IN as `(seq_len, d_model)` and come OUT as
`(seq_len, d_model)` — identical shape. The only place the shape *changes* is the intermediate
attention-weights matrix, which is `(seq_len, seq_len)`. That's the whole architecture's shape
story in one sentence, and it's exactly why you can stack as many Transformer blocks as you
want — each one's output is a valid input to the next.

### ✏️ Your Turn 2.1
For a sentence with 12 words and `d_model=64`, what shape is the attention-weights matrix?
What shape is the final output?

In [ ]:
weights_shape = None
output_shape = None
print(weights_shape, output_shape)

✅ **Solution**
```python
weights_shape = (12, 12)   # seq_len x seq_len, always -- regardless of d_model
output_shape = (12, 64)    # seq_len x d_model -- same shape the input arrived in
```

---
## Chapter 3 — Multi-Head Attention, Deep Dive: Render Each Head Separately

📖 **Theory.** A single attention computation can only capture one *kind* of relationship.
Multi-head attention runs several independent attention computations in parallel, each with
its OWN learned projections — and different heads genuinely learn to specialize. Let's not
just compute this — let's **look at each head's pattern separately** and see they're
different from each other.


In [ ]:
def multi_head_attention_with_weights(X, num_heads, d_model, seed=0):
    d_head = d_model // num_heads
    all_weights = []
    outputs = []
    for h in range(num_heads):
        rng = np.random.RandomState(seed + h)   # a DIFFERENT random projection per head
        Wq_h = rng.randn(d_model, d_head) * 0.3
        Wk_h = rng.randn(d_model, d_head) * 0.3
        Wv_h = rng.randn(d_model, d_head) * 0.3
        Qh, Kh, Vh = X @ Wq_h, X @ Wk_h, X @ Wv_h
        scores = Qh @ Kh.T / np.sqrt(d_head)
        w = np.exp(scores - scores.max(axis=1, keepdims=True))
        w /= w.sum(axis=1, keepdims=True)
        all_weights.append(w)
        outputs.append(w @ Vh)
    return outputs, all_weights

outputs, head_weights = multi_head_attention_with_weights(X, num_heads=4, d_model=8)

fig, axes = plt.subplots(1, 4, figsize=(16,4))
for i, (ax, w) in enumerate(zip(axes, head_weights)):
    im = ax.imshow(w, cmap='viridis')
    ax.set_xticks(range(seq_len)); ax.set_xticklabels(sentence, rotation=45)
    ax.set_yticks(range(seq_len)); ax.set_yticklabels(sentence if i==0 else [])
    ax.set_title(f"Head {i+1}")
plt.suptitle("4 heads on the SAME sentence -- notice each one's pattern is different")
plt.tight_layout()
plt.show()

⚡ **Pro tip.** With random (untrained) weights, heads differ but not meaningfully — in a
**trained** model, this is exactly where interpretability researchers find heads that
specialize: some heads consistently track subject-verb relationships, others track
punctuation boundaries, others track coreference (like the pronoun example in Chapter 4). The
mechanism you're looking at here is identical; only the learned weights differ.

⚠️ **Common trap.** People sometimes assume more heads is strictly better. In practice, too
many heads (each with a tiny `d_head`) gives each head too little capacity to represent
anything useful — `num_heads` is a real architectural tradeoff, not "more is free."

### ✏️ Your Turn 3.1
Re-run `multi_head_attention_with_weights` with `num_heads=8` instead of 4 (same `d_model=8`,
so each head gets `d_head=1`). Look at the resulting heatmaps — do they look more or less
structured than the 4-head version? Why might that be (hint: what can one dimension of
information usefully represent)?

In [ ]:
outputs8, head_weights8 = multi_head_attention_with_weights(X, num_heads=8, d_model=8)
# render and inspect


✅ **Solution**
```python
outputs8, head_weights8 = multi_head_attention_with_weights(X, num_heads=8, d_model=8)
# With d_head=1, each head's Q/K are single numbers -- there's almost no room to represent
# a meaningful "kind of relationship," so patterns tend to look noisier/less structured.
# This is the practical case for not maximizing num_heads blindly.
```

---
## Chapter 4 — Why Attention Works: A Real Coreference Example, Visualized

📖 **Theory.** Attention's real power: it can connect words that are **far apart** but
semantically related — like a pronoun and the noun it refers to — directly, with no
relay chain to survive. Let's build one honest, visualized example of exactly this.


In [ ]:
sentence2 = ["The", "animal", "didn't", "cross", "the", "street", "because", "it", "was", "tired"]
d = 6

# Hand-crafted embeddings (same technique as the Embeddings & Search lab): "animal" and "it"
# deliberately share a strong signal in two dimensions, simulating what a TRAINED model's
# embeddings would have learned to do on their own from data.
np.random.seed(2)
embeddings2 = {w: np.random.randn(d)*0.3 for w in set(sentence2)}
embeddings2["animal"][:2] = [2.0, 1.5]
embeddings2["it"][:2] = [1.9, 1.6]   # close to "animal" in those two dims -- simulating learned coreference

X2 = np.array([embeddings2[w] for w in sentence2])

np.random.seed(3)
Wq2 = np.eye(d) + np.random.randn(d,d)*0.05
Wk2 = np.eye(d) + np.random.randn(d,d)*0.05
Q2, K2 = X2 @ Wq2, X2 @ Wk2
scores2 = Q2 @ K2.T / np.sqrt(d)
weights2 = np.exp(scores2 - scores2.max(axis=1, keepdims=True))
weights2 /= weights2.sum(axis=1, keepdims=True)

it_idx = sentence2.index("it")
print("attention FROM 'it' TO each word in the sentence:")
for w, wt in zip(sentence2, weights2[it_idx]):
    marker = "  <---" if wt > 0.1 else ""
    print(f"  {w:10s} {wt:.3f}{marker}")

plt.figure(figsize=(8,1.5))
plt.imshow(weights2[it_idx:it_idx+1], cmap='viridis', aspect='auto')
plt.xticks(range(len(sentence2)), sentence2, rotation=45)
plt.yticks([])
plt.title("Attention FROM \'it\' -- watch it light up on \'animal\', 7 words back")
plt.colorbar(label="attention weight")
plt.show()

🧠 **Mental model.** "It" attends strongly to itself AND to "animal" — both dominate over
every other word, which is a real, honest signal (self-attention often keeps meaningful
self-weight too). Compare this to what a bag-of-words model or a heavily-diluted RNN hidden
state (Chapter 1) could do: neither has *any* mechanism to directly connect "it" back to
"animal" seven words earlier. Attention computes that connection as a **direct, first-class
number** — not something that has to survive a relay chain.

### ✏️ Your Turn 4.1
Change the sentence so the pronoun is closer to its referent (e.g. remove some of the middle
words) and re-run. Does the attention weight on the referent get stronger, weaker, or stay
about the same? What does that suggest about how distance affects attention (versus how it
affects an RNN's hidden state)?

In [ ]:
sentence3 = ["The", "animal", "was", "tired"]  # much shorter distance
# rebuild embeddings/attention for this shorter sentence and compare


✅ **Solution**
```python
# Attention weight patterns are driven by CONTENT similarity (the embeddings), not distance --
# unlike an RNN, where distance directly causes information loss (Chapter 1). Attention CAN
# still connect distant words at full strength; an RNN structurally cannot.
```

---
## Chapter 5 — Training-Time Behavior: Teacher Forcing, One Parallel Pass

📖 **Theory.** During **training**, the model already knows the correct target sequence (from
labeled data). Rather than generating one token, checking it, generating the next, etc., we
use **teacher forcing**: feed the *entire* correct sequence in at once (shifted by one
position) and compute predictions for *every* position **simultaneously**, in one matrix
operation. Combined with causal masking (from Lab 11), this trains a model to predict "the
next token" at every position at once.

🖼️ **Diagram — training computes ALL positions' predictions in one pass**
```
 input (shifted):    <s>  The  cat  sat   on
 target:              The  cat  sat   on  mat
 model predicts, for EVERY position, IN PARALLEL, in one forward pass:
   position 0: given "<s>"           -> predict "The"
   position 1: given "<s> The"       -> predict "cat"
   position 2: given "<s> The cat"   -> predict "sat"
   ... all computed in ONE matrix multiplication, not a loop
```


In [ ]:
# Demonstrate: training computes loss for ALL positions in one pass, timed against
# doing it position-by-position (which would be needlessly slow and is NOT how training works).
np.random.seed(0)
seq_len, d_model, vocab_size = 20, 32, 50
X_train = np.random.randn(seq_len, d_model)
W_out = np.random.randn(d_model, vocab_size)
targets = np.random.randint(0, vocab_size, seq_len)

import time
# the RIGHT way: one matrix multiply computes logits for every position at once
t0 = time.time()
logits_all = X_train @ W_out              # (seq_len, vocab_size) -- ALL positions, ONE matmul
t_parallel = time.time() - t0

# the WRONG (needlessly slow) way: loop over positions one at a time
t0 = time.time()
logits_loop = np.zeros((seq_len, vocab_size))
for i in range(seq_len):
    logits_loop[i] = X_train[i] @ W_out    # recompute one row at a time
t_loop = time.time() - t0

print(f"parallel (one matmul):     {t_parallel*1000:.4f} ms")
print(f"position-by-position loop: {t_loop*1000:.4f} ms")
print(f"\nsame result either way (correctness unaffected):", np.allclose(logits_all, logits_loop))
print("but training ALWAYS uses the parallel form -- across a full batch of sequences, the gap grows enormously")

⚡ **Pro tip.** This parallel-training capability is a direct consequence of Chapter 1's
"no sequential bottleneck" property. It's *the* practical reason Transformers can be trained
on massive datasets in reasonable time — GPUs are built to do millions of independent matrix
multiplications at once, and teacher-forced training is structured as exactly that.

### ✏️ Your Turn 5.1
Why can't an RNN use this same "compute every position's prediction in one parallel pass"
trick during training, even with teacher forcing?

In [ ]:
# your explanation


✅ **Solution**
```python
# An RNN's hidden state at position t REQUIRES the hidden state at position t-1 as an
# input -- there is no way to compute position 5's hidden state without first computing
# positions 0-4, in order. That sequential dependency is exactly what a Transformer's
# attention removes (Chapter 1), which is what makes the parallel training trick possible.
```

---
## Chapter 6 — Inference-Time Behavior: Autoregressive Generation & KV-Caching

📖 **Theory.** At **inference** time, there's no ground-truth "next word" to feed in — the
model must generate one token, then use ITS OWN output as input for the next step. This is
fundamentally sequential (you can't generate word 5 before word 4 exists), which is why
generation is slower than training, token by token.

**KV-caching** is the key inference optimization: the Key and Value vectors for *already
generated* tokens never change once computed — recomputing them at every new step (as a naive
implementation would) wastes enormous work. Cache them once, reuse them, and only compute the
Key/Value for the ONE new token at each step.

🖼️ **Diagram — naive vs. cached generation**
```
 NAIVE (recomputes everything every step):
   step 1: compute K,V for [tok0]                    -> generate tok1
   step 2: compute K,V for [tok0, tok1]               -> generate tok2   (tok0's K,V redone!)
   step 3: compute K,V for [tok0, tok1, tok2]         -> generate tok3   (tok0,tok1 redone!)
   total K/V computations across n steps: 1+2+3+...+n = O(n^2)

 CACHED (reuses previous K/V, computes only what's new):
   step 1: compute K,V for tok0 -> cache them          -> generate tok1
   step 2: compute K,V for tok1 ONLY -> append to cache -> generate tok2
   step 3: compute K,V for tok2 ONLY -> append to cache -> generate tok3
   total K/V computations across n steps: 1+1+1+...+1 = O(n)
```


In [ ]:
def naive_generate(n_steps, embed_table, Wq, Wk, Wv, d_model, vocab_size):
    tokens = [0]
    for _ in range(n_steps):
        X = embed_table[tokens]        # re-embed EVERY token generated so far
        K = X @ Wk; V = X @ Wv          # RECOMPUTE K, V for every token, every single step
        q = X[-1:] @ Wq
        scores = q @ K.T / np.sqrt(d_model)
        w = np.exp(scores - scores.max()); w /= w.sum()
        out = w @ V
        tokens.append(int(abs(out.sum())) % vocab_size)
    return tokens

def cached_generate(n_steps, embed_table, Wq, Wk, Wv, d_model, vocab_size):
    tokens = [0]
    K_cache = [embed_table[0] @ Wk]
    V_cache = [embed_table[0] @ Wv]
    for _ in range(n_steps):
        K = np.array(K_cache); V = np.array(V_cache)   # REUSE -- no recompute
        q = embed_table[tokens[-1]] @ Wq
        scores = (q @ K.T) / np.sqrt(d_model)
        w = np.exp(scores - scores.max()); w /= w.sum()
        out = w @ V
        next_tok = int(abs(out.sum())) % vocab_size
        tokens.append(next_tok)
        K_cache.append(embed_table[next_tok] @ Wk)      # only compute the NEW token's K,V
        V_cache.append(embed_table[next_tok] @ Wv)
    return tokens

np.random.seed(0)
d_model, vocab_size = 64, 50
embed_table = np.random.randn(vocab_size, d_model) * 0.1
Wq = np.random.randn(d_model, d_model) * 0.1
Wk = np.random.randn(d_model, d_model) * 0.1
Wv = np.random.randn(d_model, d_model) * 0.1

import time
n = 60
t0 = time.time(); naive_generate(n, embed_table, Wq, Wk, Wv, d_model, vocab_size); t_naive = time.time()-t0
t0 = time.time(); cached_generate(n, embed_table, Wq, Wk, Wv, d_model, vocab_size); t_cached = time.time()-t0
print(f"naive (recompute all K/V each step):  {t_naive*1000:.2f} ms")
print(f"cached (reuse K/V, compute only new): {t_cached*1000:.2f} ms")
print(f"measured speedup at this small scale: {t_naive/t_cached:.2f}x")

⚠️ **Honest caveat.** At this small, toy scale, Python/NumPy overhead partly masks the
underlying trend — the measured speedup here is real but modest and a bit noisy run to run.
The *reason* it matters isn't this specific number: naive generation does **O(n²)** total K/V
work across a full generation (1+2+3+...+n), while cached generation does **O(n)** (constant
work per step). At real production scale — thousands of tokens, billions of parameters — this
difference is the reason KV-caching is a *mandatory* optimization in every real LLM serving
stack (vLLM, TGI, etc.), not an optional nicety.

### ✏️ Your Turn 6.1
For `n_steps=100`, roughly how many total K/V computations does the naive approach do
(1+2+...+100), versus the cached approach? What's that ratio?

In [ ]:
n = 100
naive_total = None   # sum 1 through n
cached_total = None   # one per step
print(naive_total, cached_total)

✅ **Solution**
```python
naive_total = sum(range(1, n+1))   # 5050
cached_total = n                    # 100
# ratio: 5050/100 = 50.5x MORE total K/V computation for naive at n=100 --
# and this ratio grows linearly with n, which is exactly the O(n^2) vs O(n) gap.
```

---
## 🏆 Chapter 7 — Capstone: Verify Your Attention Matches Real PyTorch, Exactly

Same trust-building move as the micrograd lab's capstone: build multi-head attention
completely from scratch, then run the **identical weights** through PyTorch's real
`torch.nn.MultiheadAttention` and confirm the outputs match to floating-point precision.
If they don't, something in your understanding of the mechanism is wrong — this is a real
check, not a formality.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
d_model_t, num_heads_t, seq_len_t = 8, 2, 4

mha_real = nn.MultiheadAttention(embed_dim=d_model_t, num_heads=num_heads_t, batch_first=True)
x_t = torch.randn(1, seq_len_t, d_model_t)

# Extract PyTorch's REAL internal weights, so your from-scratch version uses the SAME ones
in_proj_weight = mha_real.in_proj_weight.detach()
in_proj_bias = mha_real.in_proj_bias.detach()
out_proj_weight = mha_real.out_proj.weight.detach()
out_proj_bias = mha_real.out_proj.bias.detach()
Wq_t, Wk_t, Wv_t = in_proj_weight.chunk(3, dim=0)
bq_t, bk_t, bv_t = in_proj_bias.chunk(3, dim=0)
print("Extracted PyTorch's real weights. Now build the SAME computation from scratch below.")

### ✏️ Capstone Task
Implement `scratch_mha(x, Wq, bq, Wk, bk, Wv, bv, Wo, bo, num_heads)` that reproduces
PyTorch's multi-head attention exactly: split into heads, scaled dot-product attention per
head, concatenate heads back together, final output projection. Then compare against
`mha_real`'s real output.

In [ ]:
def scratch_mha(x, Wq, bq, Wk, bk, Wv, bv, Wo, bo, num_heads):
    pass  # your implementation here

with torch.no_grad():
    real_out, _ = mha_real(x_t, x_t, x_t, need_weights=True, average_attn_weights=False)
    scratch_out = None  # call scratch_mha(...) here

max_diff = None
print(max_diff)

✅ **Capstone Solution**
```python
def scratch_mha(x, Wq, bq, Wk, bk, Wv, bv, Wo, bo, num_heads):
    B, T, D = x.shape
    d_head = D // num_heads
    Q = x @ Wq.T + bq
    K = x @ Wk.T + bk
    V = x @ Wv.T + bv
    Q = Q.view(B, T, num_heads, d_head).transpose(1, 2)
    K = K.view(B, T, num_heads, d_head).transpose(1, 2)
    V = V.view(B, T, num_heads, d_head).transpose(1, 2)
    scores = Q @ K.transpose(-2, -1) / (d_head ** 0.5)
    weights = torch.softmax(scores, dim=-1)
    out = weights @ V
    out = out.transpose(1, 2).contiguous().view(B, T, D)
    out = out @ Wo.T + bo
    return out

with torch.no_grad():
    real_out, _ = mha_real(x_t, x_t, x_t, need_weights=True, average_attn_weights=False)
    scratch_out = scratch_mha(x_t, Wq_t, bq_t, Wk_t, bk_t, Wv_t, bv_t, out_proj_weight, out_proj_bias, num_heads_t)

max_diff = (real_out - scratch_out).abs().max().item()
print("max difference:", max_diff)
assert max_diff < 1e-5
print("EXACT MATCH -- your from-scratch multi-head attention IS what PyTorch runs internally.")
```

🎉 **You now understand Transformers at the level that matters: not just the formula, but
WHY they replaced RNNs (no sequential bottleneck, no information dilution), WHAT attention
actually looks like on real data (rendered heatmaps, real coreference capture), and HOW the
exact same architecture behaves differently at training time (parallel, teacher-forced) vs.
inference time (sequential, KV-cached).** And you've proven your understanding is mechanically
identical to what a production framework runs, not an approximation of it.

---
### 📌 Concept Quick-Reference
**Why Transformers > RNNs:** no fixed-size hidden-state bottleneck (info doesn't dilute over
  distance) + fully parallelizable (no sequential step-by-step dependency) during training
**End-to-end shape story:** embeddings (seq_len, d_model) -> attention weights (seq_len,
  seq_len) -> output (seq_len, d_model) -- same shape out as in, which is why blocks stack
**Multi-head attention:** several independent attention computations in parallel; heads can
  specialize (grammar, topic, coreference) given enough capacity per head
**Why attention "works":** directly connects related words regardless of distance, driven by
  content similarity, not position -- verified visually with a real coreference example
**Training behavior:** teacher forcing + causal masking -> every position's prediction
  computed in ONE parallel pass (not a loop) -- this is what makes large-scale training feasible
**Inference behavior:** autoregressive, inherently sequential (each token needs the last) --
  KV-caching turns O(n^2) total re-computation into O(n) by reusing already-computed K/V
**Trust check:** a from-scratch implementation, given the same weights, must match a real
  framework's output exactly -- if it doesn't, the mental model has a bug somewhere
